# Python — Pièges & Astuces

Ce notebook couvre les **erreurs classiques** qui font trébucher les développeurs Python — débutants comme expérimentés. Ce sont exactement les questions qui reviennent en entretien technique.

Pour chaque piège : **lis le code, prédit le résultat, puis exécute.**

## Sommaire

1. [Mutabilité et pointeurs](#1)
2. [L'argument mutable par défaut](#2)
3. [La liste de listes partagées](#3)
4. [Modifier une liste pendant l'itération](#4)
5. [`is` vs `==` et le cache des entiers](#5)
6. [Portée des variables (scope LEGB)](#6)
7. [Late binding dans les closures](#7)
8. [Truthiness — les faux-amis de `if`](#8)
9. [Quiz](#9)

***
## 1. Mutabilité et pointeurs <a id="1"></a>

En Python, une variable est un **nom qui pointe vers un objet en mémoire**, pas une boîte qui contient une valeur. Cette distinction est source de nombreux bugs.

### 1.1 `y = x` ne crée pas une copie

⚠️ **Prédit avant d'exécuter** : après `y[0] = 42`, quelle est la valeur de `x` ?

In [ ]:
x = [1, 2, 3]
y = x        # y pointe vers le MÊME objet que x
y[0] = 42

print("y =", y)
print("x =", x)  # ?

`x` et `y` **désignent le même objet** en mémoire. Modifier `y` modifie l'objet, donc `x` « voit » le changement.

`id()` retourne l'adresse mémoire d'un objet :

In [ ]:
x = [1, 2, 3]
y = x
print(id(x))          # même adresse
print(id(y))          # même adresse
print(id(x) == id(y)) # True

### 1.2 Créer une vraie copie

Pour les types **immuables** (int, str, tuple), ce problème n'existe pas — ils ne peuvent pas être modifiés sur place.

Pour les types **mutables** (list, dict, set), utiliser `.copy()` :

In [ ]:
x = [1, 2, 3]
y = x.copy()   # nouvel objet
y[0] = 42

print("y =", y)          # [42, 2, 3]
print("x =", x)          # [1, 2, 3]  — intact
print(id(x) == id(y))    # False

### 1.3 Objets imbriqués : `.copy()` ne suffit pas

`.copy()` est une **copie superficielle** (*shallow copy*) : les objets de premier niveau sont copiés, mais les objets imbriqués restent partagés.

⚠️ **Prédit avant d'exécuter** : `x` est-il modifié ?

In [ ]:
x = [[1, 2], [3, 4]]
y = x.copy()    # copie de la liste externe...
y[0][0] = 99    # ...mais les listes internes sont partagées

print("y =", y)
print("x =", x)  # ?

Pour une copie totalement indépendante d'un objet imbriqué, utiliser `copy.deepcopy()` :

In [ ]:
import copy

x = [[1, 2], [3, 4]]
y = copy.deepcopy(x)
y[0][0] = 99

print("y =", y)   # [[99, 2], [3, 4]]
print("x =", x)   # [[1, 2], [3, 4]]  — intact

| Situation | Solution |
|---|---|
| Type immuable (int, str, tuple) | Aucune action nécessaire |
| Liste/dict à un niveau | `.copy()` |
| Objet imbriqué (liste de listes…) | `copy.deepcopy()` |

***
## 2. L'argument mutable par défaut <a id="2"></a>

C'est **le** piège classique en entretien Python. Les valeurs par défaut des arguments sont évaluées **une seule fois**, à la définition de la fonction — pas à chaque appel.

⚠️ **Prédit avant d'exécuter** : que retournent les deux appels à `ajouter(1)` ?

In [ ]:
def ajouter(element, liste=[]):   # ← liste=[] créée UNE SEULE FOIS
    liste.append(element)
    return liste

print(ajouter(1))   # ?
print(ajouter(2))   # ?  — surprise !

La liste par défaut **persiste entre les appels** : c'est un état caché dans la fonction.

On peut l'inspecter directement :

In [ ]:
print(ajouter.__defaults__)  # la liste par défaut actuelle

**La solution idiomatique** : utiliser `None` comme valeur sentinelle.

In [ ]:
def ajouter(element, liste=None):
    if liste is None:
        liste = []        # nouvelle liste à chaque appel
    liste.append(element)
    return liste

print(ajouter(1))   # [1]
print(ajouter(2))   # [2]  — comportement attendu

> Ce pattern s'applique à tout type mutable en argument par défaut : `[]`, `{}`, `set()`, objets personnalisés.

***
## 3. La liste de listes partagées <a id="3"></a>

Créer une matrice 2D avec l'opérateur `*` est tentant mais dangereux.

⚠️ **Prédit avant d'exécuter** : après avoir modifié `matrice[0][0]`, quelle est la valeur de `matrice` ?

In [ ]:
matrice = [[0] * 3] * 3    # 3 lignes de 3 zéros
print("Avant :", matrice)

matrice[0][0] = 99
print("Après :", matrice)   # ?

`[[0]*3] * 3` crée **trois références vers la même liste interne** — pas trois listes distinctes.

**Solution** : utiliser une list comprehension, qui crée un nouvel objet à chaque itération.

In [ ]:
matrice = [[0] * 3 for _ in range(3)]   # 3 listes indépendantes
matrice[0][0] = 99
print(matrice)   # [[99, 0, 0], [0, 0, 0], [0, 0, 0]]

> Règle : `[x] * n` **partage** l'objet `x` si celui-ci est mutable. `[expr for _ in range(n)]` crée `n` objets indépendants.

***
## 4. Modifier une liste pendant l'itération <a id="4"></a>

⚠️ **Prédit avant d'exécuter** : combien d'éléments contient `nombres` après la boucle ?

In [ ]:
nombres = [1, 2, 3, 4, 5, 6]

for n in nombres:
    if n % 2 == 0:          # supprimer les pairs
        nombres.remove(n)

print(nombres)   # ?

Python avance un index interne sur la liste. Quand on supprime un élément, les suivants décalent — certains sont **sautés silencieusement**.

**Solutions possibles :**

In [ ]:
# Option 1 : list comprehension (idiomatique)
nombres = [1, 2, 3, 4, 5, 6]
nombres = [n for n in nombres if n % 2 != 0]
print(nombres)   # [1, 3, 5]

In [ ]:
# Option 2 : itérer sur une copie
nombres = [1, 2, 3, 4, 5, 6]
for n in nombres.copy():
    if n % 2 == 0:
        nombres.remove(n)
print(nombres)   # [1, 3, 5]

***
## 5. `is` vs `==` et le cache des entiers <a id="5"></a>

- `==` compare les **valeurs**
- `is` compare les **identités** (adresses mémoire)

⚠️ **Prédit avant d'exécuter** : `True` ou `False` pour chaque ligne ?

In [ ]:
a = 256
b = 256
print(a == b)   # ?
print(a is b)   # ?

In [ ]:
a = 257
b = 257
print(a == b)   # ?
print(a is b)   # ?

Python **met en cache les entiers de -5 à 256** : un seul objet existe en mémoire pour ces valeurs. Au-delà, deux entiers égaux sont des objets distincts.

C'est un détail d'implémentation CPython — ne jamais écrire du code qui en dépend.

**Règle** : `is` est réservé aux comparaisons avec `None`, `True`, `False`. Pour tout le reste, utiliser `==`.

In [ ]:
# Même piège avec les chaînes
s1 = "hello"
s2 = "hello"
print(s1 is s2)    # True (interning — détail d'implémentation)

s3 = "hello world"
s4 = "hello world"
print(s3 is s4)    # Dépend de l'implémentation — ne pas compter dessus

***
## 6. Portée des variables — scope LEGB <a id="6"></a>

Python résout les noms de variables selon l'ordre **L**ocal → **E**nclosing → **G**lobal → **B**uilt-in.

⚠️ **Prédit avant d'exécuter** : ce code s'exécute-t-il sans erreur ? Que retourne-t-il ?

In [ ]:
x = 10

def incrementer():
    x += 1     # lecture ET écriture de x
    return x

incrementer()   # ?

Dès qu'une variable est **assignée** dans une fonction, Python la considère comme locale **pour toute la fonction** — y compris avant l'assignation. D'où l'`UnboundLocalError`.

Pour modifier une variable globale depuis une fonction (déconseillé — préférer un retour de valeur) :

In [ ]:
x = 10

def incrementer():
    global x    # déclare explicitement x comme globale
    x += 1
    return x

print(incrementer())  # 11
print(x)              # 11 — la variable globale a été modifiée

⚠️ **Prédit avant d'exécuter** : que retourne `lire()` ?

In [ ]:
x = 10

def lire():
    return x    # pas d'assignation → Python cherche dans le scope global

lire()   # ?

Ici pas d'erreur : `x` n'est que **lue**, donc Python remonte au scope global et trouve `10`.

> La distinction : **lire** une variable globale est permis. **Modifier** une variable globale sans `global` lève une `UnboundLocalError`.

***
## 7. Late binding dans les closures <a id="7"></a>

Les closures capturent les variables par **référence**, pas par valeur au moment de leur création.

⚠️ **Prédit avant d'exécuter** : quelle liste retourne `[f() for f in fonctions]` ?

In [ ]:
fonctions = []
for i in range(5):
    fonctions.append(lambda: i)

[f() for f in fonctions]   # ?

Toutes les lambdas capturent la **même variable `i`**. Quand elles sont appelées, la boucle est terminée et `i` vaut `4`.

**Solution** : forcer la capture de la valeur courante via un argument par défaut.

In [ ]:
fonctions = [lambda i=i: i for i in range(5)]   # i=i capture la valeur
[f() for f in fonctions]   # [0, 1, 2, 3, 4]

> Ce comportement n'est pas un bug Python — c'est la sémantique normale des closures. Il faut juste le connaître.

***
## 8. Truthiness — les faux-amis de `if` <a id="8"></a>

En Python, `if x:` ne teste pas `x == True` — il teste si `x` est **truthy**. Valeurs **falsy** : `None`, `0`, `0.0`, `""`, `[]`, `{}`, `set()`.

⚠️ **Prédit avant d'exécuter** : lesquelles de ces conditions entrent dans le `if` ?

In [ ]:
for valeur in [None, 0, 0.0, "", [], {}, False, "0", [0], 0.1]:
    if valeur:
        print(f"{repr(valeur):15} → truthy")
    else:
        print(f"{repr(valeur):15} → falsy")

### Le piège classique : `None` vs `0`

⚠️ **Prédit avant d'exécuter** : ce code se comporte-t-il comme prévu ?

In [ ]:
def traiter(valeur=None):
    if not valeur:             # ← intention : "si pas de valeur fournie"
        valeur = 42
    return valeur

print(traiter())    # 42  — ok, pas d'argument
print(traiter(10))  # 10  — ok
print(traiter(0))   # ?   — surprise !

**Solution** : tester explicitement `None` quand c'est l'intention.

In [ ]:
def traiter(valeur=None):
    if valeur is None:         # ← test précis
        valeur = 42
    return valeur

print(traiter())    # 42
print(traiter(10))  # 10
print(traiter(0))   # 0  — correct

***
## 9. Quiz <a id="9"></a>

**Q1.** Que retourne ce code ?

```python
a = [1, 2, 3]
b = a
b += [4]
print(a)
```

A) `[1, 2, 3]`  
B) `[1, 2, 3, 4]`  
C) Erreur

<details><summary>Réponse</summary>

**B) `[1, 2, 3, 4]`**

`b` et `a` pointent vers le même objet. `+=` sur une liste appelle `__iadd__` qui **mute** la liste en place (contrairement à `b = b + [4]` qui créerait un nouvel objet). Donc `a` reflète le changement.

</details>

**Q2.** Que retourne ce code ?

```python
def f(x=[]):
    x.append(1)
    return x

f()
f()
print(f())
```

A) `[1]`  
B) `[1, 1, 1]`  
C) `[1, 1]`

<details><summary>Réponse</summary>

**B) `[1, 1, 1]`**

La liste par défaut est créée une seule fois à la définition de la fonction. Elle accumule un `1` à chaque appel sans argument.

</details>

**Q3.** Que contient `m` après ce code ?

```python
m = [[0] * 2] * 2
m[0][1] = 5
print(m)
```

A) `[[0, 5], [0, 0]]`  
B) `[[0, 5], [0, 5]]`  
C) `[[5, 0], [5, 0]]`

<details><summary>Réponse</summary>

**B) `[[0, 5], [0, 5]]`**

`[[0]*2]*2` crée deux références vers la même liste interne. Modifier `m[0][1]` modifie l'unique liste, visible depuis les deux lignes.

</details>

**Q4.** Ce code lève-t-il une erreur ? Pourquoi ?

```python
compteur = 0

def incrementer():
    print(compteur)   # ligne A
    compteur += 1     # ligne B

incrementer()
```

A) Non, affiche `0` puis `compteur` vaut `1`  
B) Non, affiche `0` mais `compteur` global reste `0`  
C) Oui, `UnboundLocalError` à la ligne A

<details><summary>Réponse</summary>

**C) Oui, `UnboundLocalError` à la ligne A**

Dès que Python voit une assignation à `compteur` (ligne B), il traite `compteur` comme une variable **locale** pour toute la fonction — y compris la ligne A qui précède l'assignation. La lecture de la ligne A échoue car la variable locale n'est pas encore définie.

</details>

**Q5.** Que retourne cette expression ?

```python
funcs = [lambda: i * 2 for i in range(4)]
funcs[0]()
```

A) `0`  
B) `6`  
C) `4`  
D) `8`

<details><summary>Réponse</summary>

**B) `6`**

Late binding : toutes les lambdas capturent la même variable `i`. Une fois la compréhension terminée, `i` vaut `3`. Donc `funcs[0]()` retourne `3 * 2 = 6`, comme toutes les autres.

Correction : `[lambda i=i: i * 2 for i in range(4)]`

</details>